In [15]:
import torch as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [16]:
# This is partly based on the (more sophisticated) implementation is in the pytorch repository:
# https://github.com/pytorch/examples/blob/master/reinforcement_learning/reinforce.py
class PolicyNetwork(nn.Module):
    def __init__(self, lr, n_inputs, n_hidden, n_actions):
        super(PolicyNetwork, self).__init__()
        self.lr = lr
        self.fc1 = nn.Linear(n_inputs, n_hidden)
        self.fc2 = nn.Linear(n_hidden, n_actions)
        self.optimizer = T.optim.Adam(self.parameters(), lr=self.lr)
    
        self.device = T.device(
            'cuda:0'
            if T.cuda.is_available()
            else 'cpu:0'
        )
        self.to(self.device)
        
    def forward(self, observation):
        # ensure numpy array of float32
        observation = np.array(observation, dtype=np.float32)

        x = T.tensor(observation, dtype=T.float32).to(self.device)
        x = F.relu(self.fc1(x))
        x = F.softmax(self.fc2(x), dim=0)
        return x

In [17]:
class Agent:
    eps = np.finfo(np.float32).eps.item()

    def __init__(self, env, lr, params, gamma=0.99, epsilon=0.1):
        self.env = env
        self.gamma = gamma
        self.actions = []
        self.rewards = []
        self.policy = PolicyNetwork(
            lr=lr,
            **params
        )

    def run(self):
        observation, info = self.env.reset()
        probs = []
        rewards = []
        done = False
        t = 0

        while not done:
            action, prob = self.choose_action(observation)
            probs.append(prob)

            observation, reward, terminated, truncated, info = self.env.step(action)
            done = terminated or truncated

            rewards.append(reward)
            t += 1

        policy_loss = []
        returns = []
        R = 0
        for r in rewards[::-1]:
            R = r + self.gamma * R
            returns.insert(0, R)
        returns = T.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + self.eps)

        for log_prob, R in zip(probs, returns):
            policy_loss.append(-log_prob * R)

        if len(policy_loss) > 0:
            self.policy.optimizer.zero_grad()
            policy_loss = T.stack(policy_loss, 0).sum()
            policy_loss.backward()
            self.policy.optimizer.step()

        return t

    def choose_action(self, observation):
        output = self.policy.forward(observation)
        action_probs = T.distributions.Categorical(output)
        action = action_probs.sample()
        log_probs = action_probs.log_prob(action)
        action = action.item()
        self.actions.append(log_probs)
        return action, log_probs


In [18]:
import gymnasium as gym

env = gym.make('CartPole-v1')
env._max_episode_steps = 10000
input_dims = env.observation_space.low.reshape(-1).shape[0]
n_actions = env.action_space.n


In [19]:
import gymnasium as gym

env = gym.make('CartPole-v1')  # gym_tetris.tetris_env.TetrisEnv()
env._max_episode_steps = 10000
input_dims = env.observation_space.low.reshape(-1).shape[0]
n_actions = env.action_space.n

agent = Agent(
    env=env,
    lr=0.01,
    params=dict(n_inputs=input_dims, n_hidden=10, n_actions=n_actions),
    gamma=0.99,
)


In [22]:
update_interval = 100

scores = []
score = 0
n_episodes = 25000
stop_criterion = 1000
for i in range(n_episodes):
    #mean_score = np.mean(scores[-update_interval:])
    if len(scores) >= update_interval:
        mean_score = np.mean(scores[-update_interval:])
    else:
        mean_score = np.mean(scores) if scores else 0.0
    if (i>0) and (i % update_interval) == 0:
        print('Iteration {}, average score: {:.3f}'.format(
            i, mean_score
        ))

    score = agent.run()
    scores.append(score)
    if score >= stop_criterion:
        print('Stopping. Iteration {}, average score: {:.3f}'.format(
            i, mean_score
        ))
        break


Stopping. Iteration 1, average score: 431.000


In [23]:
%matplotlib inline

import matplotlib
from matplotlib import pyplot as plt
font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 12}

matplotlib.rc('font', **font)
plt.plot(scores)
plt.xlabel('iterations')
plt.ylabel('scores')

Text(18.722222222222214, 0.5, 'scores')

findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.
findfont: Font family 'normal' not found.


In [25]:
# import ray
# from ray import tune
# from ray.rllib.agents.ppo import PPOTrainer
# from ray.rllib.agents.dqn import DQNTrainer

# ray.init(ignore_reinit_error=True)
# trainer = PPOTrainer

# # if you run this on colab you might want to set num_workers=2
# analysis = tune.run(
#     trainer,
#     stop={'episode_reward_mean': 100},
#     config={'env': 'CartPole-v0'},
#     checkpoint_freq=1,
# )


import gymnasium as gym
from stable_baselines3 import PPO

env = gym.make("CartPole-v1")

model = PPO("MlpPolicy", env, verbose=1)

model.learn(total_timesteps=200_000)

model.save("ppo_cartpole")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 20.9     |
|    ep_rew_mean     | 20.9     |
| time/              |          |
|    fps             | 941      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 27.2        |
|    ep_rew_mean          | 27.2        |
| time/                   |             |
|    fps                  | 775         |
|    iterations           | 2           |
|    time_elapsed         | 5           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009812235 |
|    clip_fraction        | 0.125       |
|    clip_range           | 0.2         |
|    entropy_loss   